# PDF Ingestion Pipeline – SimplePaperProcessor

## What it does
Converts corporate annual report PDFs into structured, chunked JSON files ready for vector database ingestion.

## How it works
1. **PDF Parsing** – Docling `DocumentConverter` with GPU-accelerated layout analysis (`ThreadedPdfPipelineOptions`, TableFormer ACCURATE mode)
2. **Text Chunking** – Docling `HybridChunker` (max 300 tokens, 20 overlap, BGE tokenizer)
3. **Table Extraction** – HTML export + optional LLM serialization (Gemma-3-12B via LM Studio) to convert tables into natural language
4. **Section Assignment** – Positional matching of section headers to table chunks via `build_section_index()`
5. **Output** – One JSON file per PDF containing a `metainfo` block + array of chunks (text & table), saved to a timestamped output folder

## Key config
| Parameter | Value |
|---|---|
| GPU | RTX 5090 (CUDA) |
| OCR | disabled |
| Chunk size | 300 tokens |
| LLM for tables | Gemma-3-12B @ localhost:1234 |
| Test mode | first 10 PDFs |

In [1]:
import json
import datetime
import requests
from pathlib import Path
from typing import List, Dict, Optional
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.chunking import HybridChunker
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import ThreadedPdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.settings import settings

# GPU Batch-Size für Seiten-Verarbeitung
settings.perf.page_batch_size = 64


class SimplePaperProcessor:
    def __init__(self, source_dir: str, test_mode: bool = True):
        self.source_dir = Path(source_dir)
        self.test_mode = test_mode

        # Timestamp für Output-Ordner
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.output_dir = self.source_dir.parent / "docling_output" / timestamp
        self.output_dir.mkdir(exist_ok=True)

        # Test-Limit
        self.MAX_PDFS = 50 if test_mode else None

        # GPU Accelerator Setup
        accelerator_options = AcceleratorOptions(
            device=AcceleratorDevice.CUDA,
            num_threads=8
        )

        # Docling Setup
        pipeline_options = ThreadedPdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            accelerator_options=accelerator_options,
            layout_batch_size=64,
            table_batch_size=8,
        )
        pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE

        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
        )

        # Chunker
        self.chunker = HybridChunker(
            tokenizer="BAAI/bge-small-en-v1.5",
            max_tokens=300,
            overlap=20
        )

        print(f"📁 Output-Ordner: {self.output_dir.name}")
        if test_mode:
            print(f"🧪 Test-Mode: Erste {self.MAX_PDFS} PDFs")

        # LM Studio config for table serialization
        self.lm_studio_url = "http://localhost:1234/v1/chat/completions"
        self.lm_studio_model = "google/gemma-3-12b"  # Gemma 3 12B via LM Studio
        self.table_serialization = True  # Set to False to skip LLM serialization

    def serialize_table_with_llm(self, table_html: str, section: Optional[str] = None,
                                  page: Optional[int] = None) -> Optional[str]:
        """Converts an HTML table into natural language English text via local LLM (LM Studio)."""
        context_parts = []
        if section:
            context_parts.append(f"Section: {section}")
        if page:
            context_parts.append(f"Page: {page}")
        context_str = "\n".join(context_parts)

        system_prompt = (
            "You are a precise data extraction assistant. Your task is to convert HTML tables "
            "from corporate financial reports into clear, factual English prose. "
            "Rules:\n"
            "- State ALL numbers, dates, and labels exactly as they appear in the table.\n"
            "- Preserve the logical structure (e.g. row-by-row or grouped by category).\n"
            "- Do NOT add interpretation, commentary, or speculation.\n"
            "- Do NOT use markdown formatting. Output plain text only.\n"
            "- Be concise but complete — every data point in the table must appear in the output."
        )

        user_prompt = f"""Convert the following HTML table into natural language English text.

{f"Context: {context_str}" if context_str else ""}

<table>
{table_html}
</table>"""

        try:
            response = requests.post(
                self.lm_studio_url,
                json={
                    "model": self.lm_studio_model,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    "temperature": 0.1,
                    "max_tokens": 2048,
                },
                timeout=120
            )
            response.raise_for_status()
            result = response.json()
            serialized = result["choices"][0]["message"]["content"].strip()

            # Strip /no_think tags if present (Qwen3.5)
            serialized = serialized.replace("<think>", "").replace("</think>", "").strip()

            return serialized

        except requests.exceptions.ConnectionError:
            print("   ⚠️ LM Studio not reachable – skipping table serialization")
            self.table_serialization = False  # Disable for remaining tables
            return None
        except Exception as e:
            print(f"   ⚠️ LLM serialization failed: {str(e)[:80]}")
            return None

    def build_section_index(self, doc) -> List[Dict]:
        """Baut Index aller Section-Headers mit Page/Position"""
        sections = []

        for item in doc.texts:
            if hasattr(item, 'label') and str(item.label) == 'section_header':
                if hasattr(item, 'prov') and item.prov:
                    sections.append({
                        "text": item.text.strip(),
                        "page": item.prov[0].page_no,
                        "top": item.prov[0].bbox.t
                    })

        return sections

    def find_section_for_table(self, sections: List[Dict], table) -> Optional[str]:
        """Findet die Section für eine Tabelle basierend auf Position"""
        if not hasattr(table, 'prov') or not table.prov:
            return None

        table_page = table.prov[0].page_no
        table_top = table.prov[0].bbox.t

        # Finde letzte Section VOR der Tabelle
        best_section = None

        for sec in sections:
            if sec["page"] < table_page:
                best_section = sec["text"]
            elif sec["page"] == table_page and sec["top"] < table_top:
                best_section = sec["text"]

        return best_section

    def extract_chunk_metadata(self, chunk) -> Dict:
        """Extrahiert Metadaten aus einem Chunk"""
        page_no = None
        section = None

        try:
            if chunk.meta and chunk.meta.doc_items:
                for doc_item in chunk.meta.doc_items:
                    if hasattr(doc_item, 'prov') and doc_item.prov:
                        page_no = doc_item.prov[0].page_no
                        break

            if chunk.meta and chunk.meta.headings:
                section = " > ".join(chunk.meta.headings)
        except Exception:
            pass

        return {"page": page_no, "section": section}

    def load_cached_chunks(self, output_file: Path) -> Optional[List[Dict]]:
        """Lädt gecachte Chunks"""
        try:
            with open(output_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return data.get("chunks", None)
        except (json.JSONDecodeError, KeyError, IOError) as e:
            print(f"   ⚠️ Cache korrupt: {e}")
            return None

    def build_metainfo(self, pdf_name: str, doc, sections: List[Dict], all_chunks: List[Dict]) -> Dict:
        """Builds a metainfo block for the document (inspired by IBM RAG Challenge 2 pipeline)"""
        num_pages = len(doc.pages) if hasattr(doc, 'pages') and doc.pages else 0
        num_tables = len(doc.tables) if hasattr(doc, 'tables') and doc.tables else 0
        num_pictures = len(doc.pictures) if hasattr(doc, 'pictures') and doc.pictures else 0
        num_equations = len(doc.equations) if hasattr(doc, 'equations') and doc.equations else 0

        num_text_blocks = 0
        num_footnotes = 0
        if hasattr(doc, 'texts') and doc.texts:
            num_text_blocks = len(doc.texts)
            num_footnotes = sum(
                1 for t in doc.texts
                if hasattr(t, 'label') and str(t.label) == 'footnote'
            )

        num_text_chunks = sum(1 for c in all_chunks if c["type"] == "text")
        num_table_chunks = sum(1 for c in all_chunks if c["type"] == "table")
        total_tokens = sum(c.get("token_count", 0) for c in all_chunks)

        return {
            "pdf_name": pdf_name,
            "num_pages": num_pages,
            "num_sections": len(sections),
            "num_text_blocks": num_text_blocks,
            "num_tables": num_tables,
            "num_pictures": num_pictures,
            "num_equations": num_equations,
            "num_footnotes": num_footnotes,
            "num_text_chunks": num_text_chunks,
            "num_table_chunks": num_table_chunks,
            "total_chunks": num_text_chunks + num_table_chunks,
            "total_tokens": total_tokens,
        }

    def process_converted_doc(self, pdf_path: Path, result) -> Optional[List[Dict]]:
        """Verarbeitet ein konvertiertes Dokument"""
        pdf_name = pdf_path.stem
        output_file = self.output_dir / f"{pdf_name}_chunks.json"

        try:
            print(f"📄 {pdf_name}")
            doc = result.document
            all_chunks = []

            # Section-Index bauen für Tabellen
            sections = self.build_section_index(doc)
            print(f"   📑 {len(sections)} Sections gefunden")

            # Text-Chunks
            for idx, chunk in enumerate(self.chunker.chunk(doc)):
                meta = self.extract_chunk_metadata(chunk)
                all_chunks.append({
                    "chunk_id": f"{pdf_name}_text_{idx}",
                    "pdf_name": pdf_name,
                    "type": "text",
                    "page": meta["page"],
                    "section": meta["section"],
                    "token_count": self.chunker.tokenizer.count_tokens(chunk.text),
                    "text": chunk.text
                })

            # Tabellen MIT Section-Zuordnung
            if hasattr(doc, 'tables') and doc.tables:
                for idx, table in enumerate(doc.tables):
                    if hasattr(table, 'data') and table.data:
                        table_page = None
                        if hasattr(table, 'prov') and table.prov:
                            table_page = table.prov[0].page_no

                        # Section für diese Tabelle finden
                        table_section = self.find_section_for_table(sections, table)

                        table_html = table.export_to_html(doc)

                        # LLM serialization: convert HTML table to natural language
                        serialized_text = None
                        if self.table_serialization:
                            serialized_text = self.serialize_table_with_llm(
                                table_html, section=table_section, page=table_page
                            )

                        chunk_text = serialized_text if serialized_text else table_html

                        all_chunks.append({
                            "chunk_id": f"{pdf_name}_table_{idx}",
                            "pdf_name": pdf_name,
                            "type": "table",
                            "page": table_page,
                            "section": table_section,
                            "token_count": self.chunker.tokenizer.count_tokens(chunk_text),
                            "text": chunk_text,
                            "html": table_html,
                            "serialized": serialized_text is not None
                        })

            # Build metainfo
            metainfo = self.build_metainfo(pdf_name, doc, sections, all_chunks)

            # Speichern
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump({
                    "pdf_name": pdf_name,
                    "metainfo": metainfo,
                    "num_chunks": len(all_chunks),
                    "chunks": all_chunks
                }, f, indent=2, ensure_ascii=False)

            tables = sum(1 for c in all_chunks if c["type"] == "table")
            tables_with_section = sum(1 for c in all_chunks if c["type"] == "table" and c["section"])
            tables_serialized = sum(1 for c in all_chunks if c.get("serialized"))
            print(f"   ✓ {len(all_chunks)} Chunks ({tables} Tabellen, {tables_with_section} mit Section, {tables_serialized} serialized)")

            return all_chunks

        except Exception as e:
            print(f"   ❌ Fehler: {str(e)[:50]}...")
            import traceback
            traceback.print_exc()
            return None

    def process_all(self) -> Dict:
        """Verarbeitet alle PDFs"""
        pdf_files = list(self.source_dir.glob("*.pdf"))

        if self.test_mode and self.MAX_PDFS:
            pdf_files = pdf_files[:self.MAX_PDFS]
            print(f"\n🧪 TEST MODE: {len(pdf_files)} PDFs\n")
        else:
            print(f"\n📚 Verarbeite {len(pdf_files)} PDFs\n")

        stats = {"success": 0, "cached": 0, "error": 0, "total_chunks": 0}

        to_convert = pdf_files  # Alle neu für Test

        if to_convert:
            print(f"\n🚀 Batch-Conversion: {len(to_convert)} PDFs...")
            results = self.converter.convert_all(
                [str(p) for p in to_convert],
                raises_on_error=False
            )

            for pdf_path, result in zip(to_convert, results):
                if result.status != ConversionStatus.SUCCESS:
                    print(f"⚠️ {pdf_path.stem} übersprungen (Status: {result.status})")
                    stats["error"] += 1
                    continue

                chunks = self.process_converted_doc(pdf_path, result)
                if chunks:
                    stats["success"] += 1
                    stats["total_chunks"] += len(chunks)
                else:
                    stats["error"] += 1

        print(f"\n{'='*40}")
        print(f"✅ Verarbeitet: {stats['success']}")
        print(f"❌ Fehler: {stats['error']}")
        print(f"📊 Total Chunks: {stats['total_chunks']}")

        return stats


if __name__ == "__main__":
    processor = SimplePaperProcessor(
        source_dir="C:/Users/phili/PycharmProjects/UDA/data/src/fin_docs",
        test_mode=True
    )
    processor.process_all()

C:\Users\phili\PycharmProjects\UDA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-18 10:58:03,508 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-18 10:58:03,615 - INFO - Going to convert document batch...
2026-03-18 10:58:03,616 - INFO - Initializing pipeline for StandardPdfPipeline with options hash ced8fb7dc3d9965939dc2705856d65a3
2026-03-18 10:58:03,646 - INFO - Loading plugin 'docling_defaults'
2026-03-18 10:58:03,650 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-03-18 10:58:03,679 - INFO - Loading plugin 'docling_defaults'
2026-03-18 10:58:03,685 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']


📁 Output-Ordner: 20260318_105802
🧪 Test-Mode: Erste 50 PDFs

🧪 TEST MODE: 50 PDFs


🚀 Batch-Conversion: 50 PDFs...


2026-03-18 10:58:03,712 - INFO - Loading plugin 'docling_defaults'
2026-03-18 10:58:03,718 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-03-18 10:58:03,741 - INFO - Accelerator device: 'cuda:0'
2026-03-18 10:58:04,482 - INFO - Loading plugin 'docling_defaults'
2026-03-18 10:58:04,483 - INFO - Registered table structure engines: ['docling_tableformer']
2026-03-18 10:58:04,698 - INFO - Accelerator device: 'cuda:0'
2026-03-18 10:58:05,158 - INFO - Processing document CE_2012.pdf
2026-03-18 10:59:38,306 - INFO - Finished converting document CE_2012.pdf in 94.80 sec.
Token indices sequence length is longer than the specified maximum sequence length for this model (3374 > 512). Running this sequence through the model will result in indexing errors


📄 CE_2012
   📑 348 Sections gefunden


2026-03-18 11:20:21,850 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-18 11:20:21,862 - INFO - Going to convert document batch...
2026-03-18 11:20:21,865 - INFO - Processing document CE_2013.pdf


...

2026-03-18 18:36:31,757 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-03-18 18:36:31,782 - INFO - Going to convert document batch...
2026-03-18 18:36:31,786 - INFO - Processing document DISH_2002.pdf


   ✓ 1152 Chunks (145 Tabellen, 145 mit Section, 145 serialized)


2026-03-18 18:37:15,991 - INFO - Finished converting document DISH_2002.pdf in 1415.08 sec.


📄 DISH_2002
   📑 213 Sections gefunden


KeyboardInterrupt: 